# Attention CN (`attncn1`)

NCN variant with **learned dot-product attention** over common neighbours.

> AMP disabled — FP16 overflows in attention softmax.

In [ ]:
import sys, os, csv, gc
import torch

sys.path.insert(0, os.path.abspath('..'))

# PyTorch >= 2.6 changed weights_only default to True, breaking OGB dataset loading
_orig_load = torch.load
def _load_compat(f, *args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _orig_load(f, *args, **kwargs)
torch.load = _load_compat

from utils.presets import make_args, CORA, CITESEER, PUBMED, COLLAB, PPA, DDI
from utils.engine  import run_experiment, get_device, free_gpu
print('imports OK')

In [ ]:
device = get_device()
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}  |  VRAM: {props.total_memory/1e9:.1f} GB  |  Free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

## Configuration

In [ ]:
import os

PREDICTOR = 'attncn1'

CFG = dict(
    use_aa=True, use_ra=True, use_diff_feat=True,
    grad_clip=1.0,
    use_amp=False,  # AMP off — FP16 overflows in attention softmax
)

NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else os.getcwd()
CSV_PATH = os.path.join(NOTEBOOK_DIR, 'results.csv')

def save_result(dataset_name, result):
    if result is None:
        print(f'  [SKIP] {dataset_name} — experiment failed'); return

    all_metrics = result.get('all_metrics', {})
    new_rows = []
    for metric in ('Hits@20', 'Hits@50', 'Hits@100'):
        if metric not in all_metrics: continue
        m = all_metrics[metric]
        new_rows.append({'dataset': dataset_name, 'predictor': PREDICTOR,
                          'metric': metric, 'runs': len(result.get('per_run', [])),
                          'val_mean': f"{m['val_mean']:.6f}", 'val_std': f"{m['val_std']:.6f}",
                          'tst_mean': f"{m['tst_mean']:.6f}", 'tst_std': f"{m['tst_std']:.6f}"})

    existing = []
    if os.path.exists(CSV_PATH):
        with open(CSV_PATH, newline='', encoding='utf-8') as fh:
            existing = list(csv.DictReader(fh))

    # Replace stale rows for this (dataset, predictor) pair
    existing = [r for r in existing if not (r['dataset'] == dataset_name and r['predictor'] == PREDICTOR)]

    fieldnames = ['dataset','predictor','metric','runs','val_mean','val_std','tst_mean','tst_std']
    with open(CSV_PATH, 'w', newline='', encoding='utf-8') as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(existing + new_rows)

    for row in new_rows:
        print(f"  {row['metric']}: val {row['val_mean']} ±{row['val_std']}  tst {row['tst_mean']} ±{row['tst_std']}")
    print(f'  [saved → {CSV_PATH}]')

print(f'Predictor: {PREDICTOR} | Config: {CFG}')

## Experiments

### Cora

In [ ]:
args = make_args(CORA, predictor=PREDICTOR, **CFG)
args.runs            = 1
args.report_all_hits = True
args.epochs          = 100

try:
    result = run_experiment(args)
except Exception:
    import traceback; traceback.print_exc()
    result = None
finally:
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

save_result('Cora', result)

### Citeseer

In [ ]:
args = make_args(CITESEER, predictor=PREDICTOR, **CFG)
args.runs            = 1
args.report_all_hits = True
args.epochs          = 100

try:
    result = run_experiment(args)
except Exception:
    import traceback; traceback.print_exc()
    result = None
finally:
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

save_result('Citeseer', result)

### Pubmed

In [ ]:
args = make_args(PUBMED, predictor=PREDICTOR, **CFG)
args.runs            = 1
args.report_all_hits = True
args.epochs          = 100

try:
    result = run_experiment(args)
except Exception:
    import traceback; traceback.print_exc()
    result = None
finally:
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

save_result('Pubmed', result)

### collab

In [ ]:
args = make_args(COLLAB, predictor=PREDICTOR, **CFG)
args.runs            = 1
args.report_all_hits = True
args.epochs          = 50

try:
    result = run_experiment(args)
except Exception:
    import traceback; traceback.print_exc()
    result = None
finally:
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

save_result('collab', result)

### ppa

In [ ]:
args = make_args(PPA, predictor=PREDICTOR, **CFG)
args.runs            = 1
args.report_all_hits = True
args.epochs          = 50

try:
    result = run_experiment(args)
except Exception:
    import traceback; traceback.print_exc()
    result = None
finally:
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

save_result('ppa', result)

### ddi

In [ ]:
args = make_args(DDI, predictor=PREDICTOR, **CFG)
args.runs            = 1
args.report_all_hits = True
args.epochs          = 200
# DDI is large and dense — cap batch sizes and CN sampling to avoid OOM
args.batch_size = min(args.batch_size, 256)
args.testbs     = min(args.testbs, 4096)
args.cndeg      = 64
args.trndeg     = 64
args.tstdeg     = 256

try:
    result = run_experiment(args)
except Exception:
    import traceback; traceback.print_exc()
    result = None
finally:
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

save_result('ddi', result)